# 第11章　自分の施設の画像で試す ― 持ち出し、匿名化、データの整え方

**『医療診断支援AI開発　入門編 ― ゼロから動かす（入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-intro

## 11.3　匿名化は「氏名を消すこと」ではない

In [ ]:
import pydicom
from pydicom.uid import generate_uid

# ① 要素ごと削除してよいタグ
DROP = ["PatientAddress", "PatientTelephoneNumbers",
        "OtherPatientIDs", "OtherPatientIDsSequence", "OtherPatientNames",
        "PerformingPhysicianName", "OperatorsName",
        "InstitutionName", "InstitutionAddress", "InstitutionalDepartmentName",
        "StationName", "DeviceSerialNumber", "RequestedProcedureID"]

# ② 「空にする」タグ ― 規格上、要素そのものを残さなければならないもの（Type 2）を含む。
#    消してしまうと規格に合わないファイルになり、PACSや変換ツールが弾くことがある
BLANK = ["PatientBirthDate", "ReferringPhysicianName", "AccessionNumber", "StudyID",
         "StudyDate", "SeriesDate", "AcquisitionDate", "ContentDate",
         "InstanceCreationDate", "AcquisitionDateTime",
         "StudyTime", "SeriesTime", "AcquisitionTime", "ContentTime",
         "InstanceCreationTime"]          # 日付だけ消して時刻を残すと、検査台帳と突き合わせられる

UIDS = ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID",
        "FrameOfReferenceUID"]            # 位置合わせ用のUIDも施設内で追跡できる鍵になる

def anonymize(ds, new_id, uid_map):
    ds.remove_private_tags()              # 装置メーカー独自のタグに患者情報が入ることがある
    for kw in DROP:
        if kw in ds:
            delattr(ds, kw)
    for kw in BLANK:
        if kw in ds:
            setattr(ds, kw, "")
    ds.PatientID = new_id                 # 連番の研究用ID（例 "CASE0001"）
    ds.PatientName = new_id
    # UIDは「消す」のではなく「別のUIDに一貫して張り替える」。
    # 同じ元UIDには必ず同じ新UIDを割り当てないと、スライスが1検査にまとまらなくなる
    for kw in UIDS:
        if kw in ds:
            old = str(getattr(ds, kw))
            if old not in uid_map:
                uid_map[old] = generate_uid()
            setattr(ds, kw, uid_map[old])
    # ファイル先頭のヘッダにも元のUIDが写っている。ここを直さないと、保存した瞬間に漏れる
    ds.file_meta.MediaStorageSOPInstanceUID = ds.SOPInstanceUID
    # この最小例では識別情報の除去完了を確認していない
    if "PatientIdentityRemoved" in ds:
        del ds.PatientIdentityRemoved
    ds.DeidentificationMethod = "MINIMAL SCRIPT, NOT A STANDARD DEID PROFILE"
    return ds

# 使い方：uid_map は「1症例の全ファイルで一つ」を共有する
uid_map = {}
for src, dst in ファイルの対応:               # ここは自分の入出力パスに読み替える
    ds = pydicom.dcmread(src)
    anonymize(ds, "CASE0001", uid_map).save_as(dst, enforce_file_format=True)

## 11.4　PACSから出したデータを、形にする

```text
export/
├── 1.2.392.200036.9116.../     ← StudyInstanceUID がフォルダ名
│   ├── 1.2.392.../
│   │   ├── IM000001
│   │   ├── IM000002 ...        ← 拡張子すらないことがある
```

In [ ]:
import pydicom, pathlib, collections

rows, skipped = [], []
for p in pathlib.Path("export").rglob("*"):
    if not p.is_file():
        continue
    try:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
    except Exception:
        skipped.append(str(p))             # 黙って捨てない。あとで数を報告する
        continue
    rows.append({
        "path": str(p),
        "patient": getattr(ds, "PatientID", ""),
        "study": getattr(ds, "StudyInstanceUID", ""),
        "series": getattr(ds, "SeriesInstanceUID", ""),
        "desc": getattr(ds, "SeriesDescription", ""),   # 「造影後 動脈相」などが入る
        "modality": getattr(ds, "Modality", ""),
        "size": (getattr(ds, "Rows", 0), getattr(ds, "Columns", 0)),
    })

print("DICOMとして読めた %d件 / 読めなかった %d件" % (len(rows), len(skipped)))
if not rows and skipped:
    print("一つも読めませんでした。pydicom.dcmread(..., force=True) を試してください")

# シリーズごとに枚数と内容を数える。ここを見ないと、どれが使う画像か分からない
cnt = collections.Counter((r["series"], r["desc"], r["modality"], r["size"]) for r in rows)
for (ser, desc, mod, size), n in sorted(cnt.items(), key=lambda x: -x[1])[:20]:
    print("%4d枚  %-8s %-30s %s  %s" % (n, mod, desc[:30], size, ser[-12:]))

In [ ]:
import numpy as np

def load_series(paths):
    slices = [pydicom.dcmread(p) for p in paths]

    # 並べる向きは、断面の向き（ImageOrientationPatient）から求めた「面に垂直な向き」で決める。
    # 体軸座標だけで並べると、冠状断・矢状断の再構成シリーズでは全部同じ値になり、
    # 例外も出ないまま「並べ替えられていない」ボリュームができあがる
    iop = np.array(slices[0].ImageOrientationPatient, dtype=float)
    normal = np.cross(iop[:3], iop[3:])
    slices.sort(key=lambda d: float(np.dot(np.array(d.ImagePositionPatient, dtype=float), normal)))

    # 混ざりものの検出。実データでは、同じフォルダにスカウト像や線量レポートが紛れる
    sizes = {(d.Rows, d.Columns) for d in slices}
    assert len(sizes) == 1, "画像サイズが混在しています: %s。シリーズを選び直してください" % sizes

    vol = np.stack([d.pixel_array for d in slices]).astype("float32")
    # slope/intercept はスライスごとに違うことがあるので、1枚ずつ掛ける
    for i, d in enumerate(slices):
        vol[i] = vol[i] * float(getattr(d, "RescaleSlope", 1.0)) \
                        + float(getattr(d, "RescaleIntercept", 0.0))

    # 1ボクセルが何ミリかを、必ず一緒に持ち帰る（画素値だけでは病変の大きさが分からない）
    py, px = [float(v) for v in slices[0].PixelSpacing]        # 面内の1画素の実寸[mm]
    # 1枚だけでは隣接断面の間隔を確認できない。SliceThickness や 1 mm で代用しない（本文の前提どおり）
    assert len(slices) > 1, "断面が1枚しかありません。間隔（dz）を確認できないので、シリーズを選び直してください"
    dz = abs(float(np.dot(np.array(slices[1].ImagePositionPatient, dtype=float)
                          - np.array(slices[0].ImagePositionPatient, dtype=float), normal)))
    return vol, (dz, py, px)               # CTならこれでHU値になる（第8章）

vol, spacing = load_series(paths)   # paths は、上の一覧で選んだ1シリーズ分のファイルパスの一覧に読み替える
np.savez("data/CASE0001.npz", vol=vol, spacing=spacing)   # 実寸を捨てずに保存する

## 11.5　正解ラベルを作る ― ここだけは、人の仕事

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "case_id":    ["CASE0001", "CASE0002", "CASE0003"],
    "patient_id": ["P001",     "P001",     "P002"],     # 同一患者の別検査に注意
    "path":       ["data/CASE0001.npz", "data/CASE0002.npz", "data/CASE0003.npz"],
    "label":      [1, 1, 0],
    "hold":       [0, 1, 0],       # 判断に迷ったら1。学習からは外し、あとで見直す
    "note":       ["", "境界例", ""],
})
# encoding を指定しないと、日本語WindowsのExcelで note 列が文字化けする
df.to_csv("dataset.csv", index=False, encoding="utf-8-sig")

# 同一患者が複数症例を持つときの分割は、必ず patient_id 単位で（第10章の鉄則）
print(df.groupby("patient_id").size())